# ДЗ 4. DML та DDL команди. Складні SQL-вирази

**Датасет:** Brazilian E-Commerce Public Dataset by Olist — [kaggle.com/datasets/olistbr/brazilian-ecommerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
Дані завантажуються через `kagglehub` (публічний датасет, ключ Kaggle не потрібен).

## 1. Setup і typed-схема Olist

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

pkg = "pgserver" if sys.version_info < (3, 13) else "pixeltable-pgserver"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       pkg, "jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "kagglehub"])

import pandas as pd
from sqlalchemy import create_engine, text

pgdata = Path("/tmp/hw4_pg")
if pkg == "pgserver":
    import pgserver
    pg = pgserver.get_server(pgdata, cleanup_mode="stop")
else:
    import pixeltable_pgserver
    pg = pixeltable_pgserver.get_server(pgdata, postgres_version=16)

engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 0
%config SqlMagic.feedback = 0
%sql engine

PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [2]:
import kagglehub

dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(dataset_dir)

/home/user/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


### Raw-таблиці

Raw-шар повторює CSV як є і потрібен лише як проміжний етап перед typed-схемою.

In [3]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

for table_name, file_name in files.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


olist_order_reviews_raw (99224, 7)


### Typed-таблиці

In [4]:
%%sql
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN ('created', 'approved', 'invoiced', 'processing',
                            'shipped', 'delivered', 'unavailable', 'canceled'))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id           BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id               TEXT,
    order_id                TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score            SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title    TEXT,
    review_comment_message  TEXT,
    review_creation_date    TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews (order_id);

SELECT t.table_name, COUNT(c.column_name) AS n_columns
FROM information_schema.tables AS t
JOIN information_schema.columns AS c ON c.table_name = t.table_name AND c.table_schema = t.table_schema
WHERE t.table_schema = 'public'
  AND t.table_name LIKE 'olist%'
  AND t.table_name NOT LIKE '%raw'
GROUP BY t.table_name
ORDER BY t.table_name;

table_name,n_columns
olist_customers,5
olist_order_items,7
olist_order_reviews,8
olist_orders,8
olist_products,9
olist_sellers,4


Перенесення raw → typed і кількість рядків у raw-шарі. У CSV колонки `product_name_lenght` і `product_description_lenght` написані з помилкою, у typed-схемі вони нормалізовані.

In [5]:
%%sql
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP,
       order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP,
       order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER,
       product_description_lenght::INTEGER,
       product_photos_qty::INTEGER,
       product_weight_g::INTEGER,
       product_length_cm::INTEGER,
       product_height_cm::INTEGER,
       product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP,
       price::NUMERIC(12, 2),
       freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id, order_id, review_score, review_comment_title,
    review_comment_message, review_creation_date, review_answer_timestamp
)
SELECT review_id, order_id, review_score::SMALLINT, review_comment_title,
       review_comment_message, review_creation_date::TIMESTAMP, review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;

SELECT 'customers' AS table_name, COUNT(*) AS n_raw FROM olist_customers_raw
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews_raw
ORDER BY table_name;

table_name,n_raw
customers,99441
order_items,112650
orders,99441
products,32951
reviews,99224
sellers,3095


### Smoke-test кількості рядків

In [6]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;

table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
reviews,99224
sellers,3095


**Interpretation:** Кількість рядків у typed-таблицях повністю збігається з raw: 99 441 клієнтів і замовлень, 112 650 позицій, 32 951 товар, 3 095 продавців, 99 224 відгуки. Жоден рядок не загубився: усі FK знайшли батьківські записи, приведення типів пройшло, а `CHECK` не відхилив жодного значення. У `olist_order_reviews` використано surrogate key `review_row_id`, бо `review_id` у джерелі не унікальний (див. Data-quality summary).

## 2. DML-запити з PostgreSQL-фічами

### 2.1. `INSERT ... RETURNING`: alert для продавця з найбільшою кількістю запізнень

In [7]:
%%sql
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT seller_id, 'late_delivery', 4,
       'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;

alert_id,seller_id,alert_type,severity,details,created_at
1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-10-17 08:49:50.871906+00:00


**Interpretation:** Створено один alert: продавець `4a3ca931…` має 172 замовлення, доставлені пізніше за обіцяну дату. `RETURNING` одразу показує `alert_id`, `created_at` і текст alert-у без окремого `SELECT`, тому результат клітинки є audit-log того, що саме записано. Запит не ідемпотентний: кожен запуск додає новий alert, тому таблицю перестворюємо на початку клітинки.

### 2.2. `INSERT ... ON CONFLICT DO UPDATE`: idempotent upsert seller-score

Upsert обгорнуто в CTE: `RETURNING` повертає по рядку на кожного продавця (~3 тис.), тому показуємо його підсумок за `tier`. `xmax = 0` у `RETURNING` означає, що рядок щойно вставлено, а не оновлено.

In [8]:
%%sql
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id  TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating NUMERIC(3, 2),
    n_reviews  INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier       TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
),
upserted AS (
    INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
    SELECT seller_id, avg_rating, n_reviews,
           CASE
               WHEN avg_rating IS NULL THEN NULL
               WHEN avg_rating >= 4.5 THEN 'gold'
               WHEN avg_rating >= 3.5 THEN 'silver'
               ELSE 'bronze'
           END AS tier
    FROM seller_rating
    ON CONFLICT (seller_id) DO UPDATE
    SET avg_rating = EXCLUDED.avg_rating,
        n_reviews  = EXCLUDED.n_reviews,
        tier       = EXCLUDED.tier,
        updated_at = NOW()
    RETURNING seller_id, avg_rating, n_reviews, tier, updated_at, (xmax = 0) AS inserted
)
SELECT tier,
       COUNT(*) AS n_sellers,
       COUNT(*) FILTER (WHERE inserted) AS n_inserted,
       COUNT(*) FILTER (WHERE NOT inserted) AS n_updated,
       ROUND(AVG(avg_rating), 2) AS avg_rating,
       SUM(n_reviews) AS n_reviews
FROM upserted
GROUP BY tier
ORDER BY tier NULLS LAST;

tier,n_sellers,n_inserted,n_updated,avg_rating,n_reviews
bronze,539,539,0,2.30,5031
gold,983,983,0,4.83,7093
silver,1568,1568,0,4.06,87123
None,5,5,0,None,0


Повторний запуск того самого upsert — кількість рядків у `seller_score` не змінюється:

In [9]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
),
upserted AS (
    INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
    SELECT seller_id, avg_rating, n_reviews,
           CASE
               WHEN avg_rating IS NULL THEN NULL
               WHEN avg_rating >= 4.5 THEN 'gold'
               WHEN avg_rating >= 3.5 THEN 'silver'
               ELSE 'bronze'
           END AS tier
    FROM seller_rating
    ON CONFLICT (seller_id) DO UPDATE
    SET avg_rating = EXCLUDED.avg_rating,
        n_reviews  = EXCLUDED.n_reviews,
        tier       = EXCLUDED.tier,
        updated_at = NOW()
    RETURNING seller_id, avg_rating, n_reviews, tier, updated_at, (xmax = 0) AS inserted
)
SELECT tier,
       COUNT(*) AS n_sellers,
       COUNT(*) FILTER (WHERE inserted) AS n_inserted,
       COUNT(*) FILTER (WHERE NOT inserted) AS n_updated,
       ROUND(AVG(avg_rating), 2) AS avg_rating,
       SUM(n_reviews) AS n_reviews
FROM upserted
GROUP BY tier
ORDER BY tier NULLS LAST;

tier,n_sellers,n_inserted,n_updated,avg_rating,n_reviews
bronze,539,0,539,2.30,5031
gold,983,0,983,4.83,7093
silver,1568,0,1568,4.06,87123
None,5,0,5,None,0


In [10]:
%%sql
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT seller_id) AS n_sellers FROM seller_score;

n_rows,n_sellers
3095,3095


**Interpretation:** Upsert записав 3 095 продавців: 983 `gold`, 1 568 `silver`, 539 `bronze` і 5 без `tier`.

- `seller_orders` використовує `SELECT DISTINCT`, бо в одному замовленні може бути кілька позицій одного продавця; без цього замовлення рахувалося б кілька разів.
- Review спочатку агрегується до `order_id`: інакше join з `order_items` повторив би той самий відгук для кожної позиції і завищив вагу великих замовлень.
- `ON CONFLICT (seller_id) DO UPDATE` робить запуск повторюваним: другий запуск дав `n_inserted = 0` і `n_updated = 3 095`, а в таблиці так само 3 095 рядків, без дублікатів.
- 5 продавців мають `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`: їхні замовлення не мають жодного відгуку, тож `LEFT JOIN` залишає їх у таблиці, але середнє рахувати нема з чого.

Обмеження: review в Olist прив’язаний до замовлення, а не до продавця. У 1 278 замовленнях кілька продавців (див. Data-quality summary), і той самий відгук зараховується кожному з них. Для навчальної задачі це прийнятне спрощення.

### 2.3. `UPDATE ... FROM`: прапорець запізнення в `olist_orders`

Тут `RETURNING` повертає ~99 тис. рядків, тому так само показуємо підсумок.

In [11]:
%%sql
ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT order_id,
           (order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) AS is_late_calc
    FROM olist_orders
),
updated AS (
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
    RETURNING o.order_id, o.customer_id, o.is_late
)
SELECT is_late, COUNT(*) AS n_orders,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_orders
FROM updated
GROUP BY is_late
ORDER BY is_late;

is_late,n_orders,pct_orders
False,92906,93.43
True,6535,6.57


**Interpretation:** Прапорець `is_late = TRUE` отримали 6 535 замовлень (6.57%), решта 92 906 — `FALSE`. Значення перераховується з дат щоразу, тому повторний запуск дасть той самий результат, а не допише `TRUE` поверх старих значень. Замовлення без фактичної або обіцяної дати доставки вважаються не простроченими, бо для них запізнення не можна визначити.

### 2.4. `DELETE ... RETURNING`: видалення застарілих alert-ів

In [12]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

alert_id,seller_id,alert_type,created_at


**Interpretation:** Результат порожній, і це очікувано: alert створено кілька секунд тому в п. 2.1, тому жоден не старший за 30 днів. `RETURNING` тут працює як журнал видалення: якби рядки видалялися, ми побачили б, які саме, ще до того, як вони зникли.

## 3. `customer_segments`

### Customer dimension

In [13]:
%%sql
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (customer_unique_id, latest_customer_id, customer_state, customer_city)
SELECT DISTINCT ON (c.customer_unique_id)
       c.customer_unique_id,
       c.customer_id AS latest_customer_id,
       c.customer_state,
       c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id;

SELECT (SELECT COUNT(*) FROM olist_customers)    AS n_customer_ids,
       (SELECT COUNT(*) FROM olist_customer_dim) AS n_unique_customers;

n_customer_ids,n_unique_customers
99441,96096


### DDL і перевірка constraints

In [14]:
%%sql
CREATE TABLE customer_segments (
    segment_id         BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    segment_name       TEXT NOT NULL CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
    rfm_score          SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
    recency_days       INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
    monetary_value     NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
    n_orders           INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
    assigned_at        TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;

SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;

conname,contype,definition
customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
customer_segments_pkey,p,PRIMARY KEY (segment_id)
customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'new'::text, 'inactive'::text])))"
fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE


### Наповнення через `INSERT ... SELECT ... ON CONFLICT`

`RETURNING` повертає ~96 тис. рядків, тому показуємо розподіл сегментів.

In [15]:
%%sql
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
           MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
           END AS recency_days,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 'VIP'
               WHEN cs.last_order_date IS NOT NULL
                    AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                    AND cs.n_orders = 1 THEN 'new'
               WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
               ELSE 'inactive'
           END AS segment_name,
           CASE
               WHEN cs.monetary_value >= 1000 THEN 5
               WHEN cs.monetary_value >= 500 THEN 4
               WHEN cs.n_orders >= 2 THEN 3
               WHEN cs.n_orders = 1 THEN 2
               ELSE 1
           END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
),
upserted AS (
    INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
    SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
    FROM segmented
    ON CONFLICT (customer_unique_id) DO UPDATE
    SET segment_name   = EXCLUDED.segment_name,
        rfm_score      = EXCLUDED.rfm_score,
        recency_days   = EXCLUDED.recency_days,
        monetary_value = EXCLUDED.monetary_value,
        n_orders       = EXCLUDED.n_orders,
        assigned_at    = NOW()
    RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
)
SELECT segment_name,
       COUNT(*) AS n_customers,
       ROUND(AVG(n_orders), 2) AS avg_orders,
       ROUND(AVG(monetary_value), 2) AS avg_monetary,
       ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM upserted
GROUP BY segment_name
ORDER BY n_customers DESC;

segment_name,n_customers,avg_orders,avg_monetary,avg_recency_days
regular,45982,1.07,224.57,307
inactive,40032,1.00,59.28,316
new,8891,1.00,140.39,72
VIP,1191,1.11,1591.81,290


**Interpretation:** У `olist_customers` 99 441 `customer_id`, але лише 96 096 реальних покупців (`customer_unique_id`). Сегментація: 45 982 `regular`, 40 032 `inactive`, 8 891 `new` і 1 191 `VIP` із середньою сумою 1 591.81 BRL. Більшість має одне замовлення (середнє 1.00–1.11), тому сегменти фактично визначаються сумою та давністю покупки.

- `customer_unique_id` використано, бо в Olist `customer_id` створюється для кожного замовлення; з ним `n_orders` майже завжди дорівнював би 1, а повторні покупки однієї людини розбилися б на різні рядки.
- `olist_customer_dim` потрібна для FK: у `olist_customers` `customer_unique_id` не унікальний, а FK може посилатися лише на `PRIMARY KEY` або `UNIQUE`-колонку.
- `reference_date` береться з датасету (день після останнього замовлення, 2018-10-18), а не з `CURRENT_DATE`: дані історичні, і з поточною датою `recency_days` росли б щодня, а сегменти змінювалися б без жодних нових даних.
- `ON CONFLICT (customer_unique_id) DO UPDATE` оновлює наявний рядок замість помилки `UNIQUE`, тому сегментацію можна перераховувати скільки завгодно разів.

Вивід constraints підтверджує PK, `UNIQUE`, FK з `ON DELETE CASCADE` і п’ять `CHECK`.

## 4. JOIN-запити

### 4.1. `INNER JOIN`: топ категорій за revenue і штатом продавця

**Business-question:** Які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [16]:
%%sql
SELECT p.product_category_name,
       s.seller_state,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o ON o.order_id = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_sellers AS s ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;

product_category_name,seller_state,n_orders,revenue
cama_mesa_banho,SP,8214,1070048.14
relogios_presentes,SP,4494,1016168.80
beleza_saude,SP,5684,790967.66
esporte_lazer,SP,4772,680165.54
moveis_decoracao,SP,4597,605650.63
cool_stuff,SP,2621,484196.53
utilidades_domesticas,SP,4021,471810.66
automotivo,SP,3018,438833.22
informatica_acessorios,SP,2783,388506.42
bebes,SP,1981,339129.74


**Interpretation:** Усі 10 найприбутковіших комбінацій — продавці з SP. Лідери — `cama_mesa_banho` (1.07 млн BRL, 8 214 замовлень) і `relogios_presentes` (1.02 млн BRL при майже вдвічі меншій кількості замовлень, тобто дорожчий середній чек). Ринок продавців сильно сконцентрований у Сан-Паулу, тому штат продавця як ML-feature буде незбалансованим.

### 4.2. `LEFT JOIN` + `COALESCE`: customer-level features

**Business-question:** Які customer-level features можна побудувати для реального покупця на рівні `customer_unique_id`?

In [17]:
%%sql
WITH order_totals AS (
    SELECT o.order_id, c.customer_unique_id,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT ot.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id,
       cd.customer_state,
       COALESCE(cs.n_orders, 0) AS n_orders,
       COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
       COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;

customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


**Interpretation:** Найбільший покупець витратив 13 664.08 BRL за одне замовлення і поставив оцінку 1, тобто великий чек не гарантує задоволення. Серед топ-20 майже всі мають 1 замовлення, лише кілька — 2 або 4. `review_per_order` спочатку зводить відгуки до рівня замовлення, тому оцінка не множиться на кількість позицій. `COALESCE(avg_review_score, 0)` дає 0 покупцям без відгуків (як `459bef48…`); для моделі краще окремий прапорець `has_review`, бо 0 змішується з поганою оцінкою.

### 4.3. `FULL OUTER JOIN`: штати клієнтів і продавців

**Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [18]:
%%sql
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
),
seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT COALESCE(c.state, s.state) AS state,
       c.n_customers,
       s.n_sellers,
       CASE
           WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
           WHEN c.state IS NOT NULL THEN 'customers_only'
           ELSE 'sellers_only'
       END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s ON s.state = c.state
ORDER BY state;

state,n_customers,n_sellers,state_presence
AC,81,1,both
AL,413,None,customers_only
AM,148,1,both
AP,68,None,customers_only
BA,3380,19,both
CE,1336,13,both
DF,2140,30,both
ES,2033,23,both
GO,2020,40,both
MA,747,1,both


**Interpretation:** Клієнти є у всіх 27 штатах, а продавці — у 23. Чотири штати (AL, AP, RR, TO) мають лише клієнтів, `sellers_only` немає. Навіть у великих штатах продавців мало: у RJ 12 852 клієнти і 171 продавець, тоді як у SP 1 849 продавців. Покупці з півночі й північного сходу отримують товари з інших штатів, що пояснює вищу доставку і запізнення.

### 4.4. `SELF JOIN`: пари продавців з одного штату, але з різних міст

**Business-question:** Які пари продавців можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [19]:
%%sql
SELECT s1.seller_id AS seller_a,
       s2.seller_id AS seller_b,
       s1.seller_state,
       s1.seller_city AS city_a,
       s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
  ON s1.seller_state = s2.seller_state
 AND s1.seller_id < s2.seller_id
 AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;

seller_a,seller_b,seller_state,city_a,city_b
1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


**Interpretation:** Запит знаходить пари продавців з одного штату, але з різних міст; `s1.seller_id < s2.seller_id` прибирає дзеркальні пари й пару продавця із самим собою. Таких пар дуже багато, тому показано лише перші 20 за сортуванням (штат BA). Видно проблему якості: місто записане як `bahia` (назва штату) і `arraial d'ajuda (porto seguro)`, тобто назви міст не нормалізовані.

### 4.5. Anti-join: замовлення без відгуку

**Business-question:** Скільки замовлень не мають жодного review-запису і в яких вони статусах?

In [20]:
%%sql
SELECT o.order_status,
       COUNT(*) AS n_orders,
       COUNT(*) FILTER (WHERE NOT EXISTS (
           SELECT 1 FROM olist_order_reviews AS rv WHERE rv.order_id = o.order_id
       )) AS n_without_review
FROM olist_orders AS o
GROUP BY o.order_status
ORDER BY n_without_review DESC, o.order_status;

order_status,n_orders,n_without_review
delivered,96478,646
shipped,1107,75
canceled,625,20
unavailable,609,14
processing,301,6
invoiced,314,5
created,5,2
approved,2,0


**Interpretation:** Без відгуку 768 замовлень із 99 441 (≈0.8%). Найбільше їх серед `delivered` (646, ≈0.7%), але відносно частіше відгук відсутній у `shipped` (75 з 1 107) — товар ще не отримано. `NOT EXISTS` перевіряє лише факт наявності відгуку і не множить рядки, на відміну від звичайного `JOIN`.

### 4.5 (продовження). Приклади замовлень без відгуку через `LEFT JOIN ... IS NULL`

**Business-question:** Які саме замовлення першими потрапили в дані без жодного відгуку?

In [21]:
%%sql
SELECT o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 10;

order_id,customer_id,order_status,order_purchase_timestamp
cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


**Interpretation:** Той самий anti-join через `LEFT JOIN ... WHERE rv.review_row_id IS NULL` показує найстаріші замовлення без відгуку: вони з жовтня 2016 – січня 2017, переважно `delivered`. Серед них є `shipped`-замовлення 2016 року, які так і не отримали статус доставки, — це ще одна ознака неповних даних на початку періоду.

## 5. `GROUP BY`, `HAVING` і агрегати

### 5.1. Revenue за категоріями з фільтром `HAVING`

**Business-question:** Які категорії товарів дають понад 100 000 BRL gross revenue і яка в них середня ціна позиції?

In [22]:
%%sql
SELECT p.product_category_name,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
       SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
       AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 100000
ORDER BY gross_revenue DESC NULLS LAST;

product_category_name,n_orders,gross_revenue,freight,avg_item_price
beleza_saude,8836,1258681.34,182566.73,130.16
relogios_presentes,5624,1205005.68,100535.93,201.14
cama_mesa_banho,9417,1036988.68,204693.04,93.30
esporte_lazer,7720,988048.97,168607.51,114.34
informatica_acessorios,6689,911954.32,147318.08,116.51
moveis_decoracao,6449,729762.49,172749.30,87.56
cool_stuff,3632,635290.85,84039.10,167.36
utilidades_domesticas,5884,632248.66,146149.11,90.79
automotivo,3897,592720.11,92664.21,139.96
ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** Понад 100 000 BRL gross revenue мають 27 категорій; лідери — `beleza_saude` (1.26 млн) і `relogios_presentes` (1.21 млн). `pcs` має лише 181 замовлення, але найвищу середню ціну позиції 1 098.34 BRL. Окремим рядком іде `None` — 179 535 BRL у товарах без категорії, які не можна просто відкинути.

### 5.2. AOV за штатом покупця для штатів із ≥ 100 замовленнями

**Business-question:** У яких штатах середній чек (AOV) найвищий, якщо брати лише штати з достатньою кількістю замовлень?

In [23]:
%%sql
WITH order_totals AS (
    SELECT o.order_id, c.customer_state,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, c.customer_state
)
SELECT customer_state,
       COUNT(*) AS n_orders,
       SUM(order_total)::NUMERIC(12, 2) AS gmv,
       (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;

customer_state,n_orders,gmv,aov
PB,517,137838.55,266.61
AL,397,94172.49,237.21
RO,243,56966.00,234.43
PA,946,212023.57,224.13
PI,476,105178.19,220.96
TO,274,60007.37,219.01
RN,474,100714.78,212.48
SE,335,70289.13,209.82
CE,1279,266436.77,208.32
MA,717,147803.55,206.14


**Interpretation:** Найвищий середній чек мають невеликі штати півночі й північного сходу: PB (266.61 BRL), AL (237.21), RO (234.43); найнижчий — SP (142.46), який дає найбільший GMV (5.77 млн). Ймовірно, у віддалених штатах вища доставка і купують дорожчі товари. Різниця між `WHERE` і `HAVING` видна саме тут: `WHERE o.order_status = 'delivered'` відкидає окремі замовлення ще до групування, а `HAVING COUNT(*) >= 100` відкидає цілі штати вже після підрахунку — так з результату випали AC, AP і RR.

### 5.3. Категорії повторних покупців через `STRING_AGG`

**Business-question:** Що купують покупці, які зробили 3 і більше замовлень?

In [24]:
%%sql
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
       STRING_AGG(DISTINCT COALESCE(p.product_category_name, '(unknown)'), ', '
                  ORDER BY COALESCE(p.product_category_name, '(unknown)')) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 3
ORDER BY n_orders DESC, total_spend DESC
LIMIT 15;

customer_unique_id,n_orders,total_spend,categories
8d50f5eadf50201ccdcedfb9e2ac8455,16,729.62,"(unknown), construcao_ferramentas_ferramentas, esporte_lazer, fashion_bolsas_e_acessorios"
3e43e6105506432c953e165fb2acf44c,9,1000.85,"cama_mesa_banho, casa_construcao, informatica_acessorios, moveis_decoracao, utilidades_domesticas"
1b6c7548a2a1f9037c1fd3ddfed95f33,7,809.21,"beleza_saude, moveis_decoracao, papelaria, perfumaria, utilidades_domesticas"
ca77025e7201e3b30c44b472ff346268,7,806.61,"cama_mesa_banho, cool_stuff, moveis_decoracao, utilidades_domesticas"
6469f99c1f9dfae7733b25662e7f1782,7,664.20,cama_mesa_banho
47c1a3033b8b77b3ab6e109eb4d5fdf3,6,836.60,"automotivo, beleza_saude, cama_mesa_banho, cool_stuff, tablets_impressao_imagem"
dc813062e0fc23409cd255f7f53c7074,6,687.76,"brinquedos, cool_stuff, relogios_presentes"
63cfc61cee11cbe306bff5857d00bfe4,6,579.81,"(unknown), cama_mesa_banho, construcao_ferramentas_construcao, moveis_decoracao, moveis_sala"
f0e310a6839dce9de1638e0fe5ab282a,6,438.09,"(unknown), cama_mesa_banho, utilidades_domesticas"
12f5d6e1cbf93dafd9dcc19095df0b3d,6,58.40,"fashion_bolsas_e_acessorios, moveis_quarto, pet_shop"


**Interpretation:** Найактивніший покупець зробив 16 замовлень з позиціями (усього 17, одне без позицій) на 729.62 BRL у чотирьох категоріях, включно з `(unknown)`. Повторні покупці часто повертаються до домашніх категорій (`cama_mesa_banho`, `utilidades_domesticas`, `moveis_decoracao`). `STRING_AGG(DISTINCT ...)` згортає категорії в один рядок, що зручно як текстова feature або для перевірки вручну.

## 6. Set operations і window function

### 6.1. `INTERSECT`: штати, де є і клієнти, і продавці

**Business-question:** У яких штатах присутні одночасно і покупці, і продавці?

In [25]:
%%sql
SELECT customer_state AS state FROM olist_customers
INTERSECT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state;

state
AC
AM
BA
CE
DF
ES
GO
MA
MG
MS


**Interpretation:** Покупці й продавці одночасно є у 23 штатах. Обрано `INTERSECT`, бо питання саме про перетин двох наборів; він також прибирає дублікати, тому окремий `DISTINCT` не потрібен. Результат збігається з групою `both` у `FULL OUTER JOIN` із п. 4.3.

### 6.2. `EXCEPT`: штати з клієнтами, але без продавців

**Business-question:** У яких штатах є покупці, але немає жодного продавця?

In [26]:
%%sql
SELECT customer_state AS state FROM olist_customers
EXCEPT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state;

state
AL
AP
RR
TO


**Interpretation:** `EXCEPT` повертає штати, де є покупці, але немає продавців: AL, AP, RR, TO. Це ті самі `customers_only` із п. 4.3, тож set-оператор дає ту саму відповідь коротшим запитом. `EXCEPT` обрано, бо потрібна різниця множин, а не об’єднання чи перетин.

### 6.3. Window function: найновіше замовлення кожного покупця через `ROW_NUMBER()`

**Business-question:** Яке останнє замовлення у покупців з найбільшою кількістю замовлень?

In [27]:
%%sql
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.customer_id, o.order_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                              ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC) AS rn,
           COUNT(*) OVER (PARTITION BY c.customer_unique_id) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
)
SELECT customer_unique_id, customer_id, order_id, order_purchase_timestamp, n_orders
FROM ranked_orders
WHERE rn = 1
ORDER BY n_orders DESC, customer_unique_id
LIMIT 10;

customer_unique_id,customer_id,order_id,order_purchase_timestamp,n_orders
8d50f5eadf50201ccdcedfb9e2ac8455,a682769c4bc10fc6ef2101337a6c83c9,d3582fd5ccccd9cb229a63dfb417c86f,2018-08-20 19:14:26,17
3e43e6105506432c953e165fb2acf44c,1815fdaaae692479110428598596b5e5,ff89ef7b3952bba5ac06d61c4a79ffbe,2018-02-27 18:36:39,9
1b6c7548a2a1f9037c1fd3ddfed95f33,fff7466a253c0e59499ea943462c10f9,43f08913407cac4e74a898d968e58c1a,2018-02-14 13:22:12,7
6469f99c1f9dfae7733b25662e7f1782,4478700ff008d3fffa02e95d89b9795f,ccb1046449f1a494de800567e7cb5979,2018-06-28 00:43:34,7
ca77025e7201e3b30c44b472ff346268,6ccedfba5919d72fcc8c51bfa982de62,8dacd1da7c4dc40d5c59e7ddd93b3ac8,2018-06-01 11:38:29,7
12f5d6e1cbf93dafd9dcc19095df0b3d,ed1793d2d1e4175d5846ce7ebb4a01f5,747996a66f5aa711deb8ae58f5ae46a0,2017-01-05 15:25:10,6
47c1a3033b8b77b3ab6e109eb4d5fdf3,5daf8c734e89f88e5305f9283e7ca135,b86ec88f5bffbf1f61fb5daed332e355,2018-01-24 15:15:26,6
63cfc61cee11cbe306bff5857d00bfe4,906614004e38a1e922ff033563bfdcf2,c29713133e5e33937af1a6e1e1370552,2018-05-28 17:20:02,6
dc813062e0fc23409cd255f7f53c7074,4fd0b9bb860c9a3d841746e1cb742314,f0031384d36ff045ee1e866942d0fa7a,2018-08-23 00:07:26,6
de34b16117594161a6a89c50b289d35a,cbbd508281ea66e2f36b4acfc28a85b0,d29eca3633b0b60adf268cae9133cd93,2018-01-17 23:34:21,6


**Interpretation:** `ROW_NUMBER()` нумерує замовлення кожного покупця від найновішого, і `rn = 1` залишає лише останнє. `COUNT(*) OVER` у тому самому вікні дає кількість замовлень без окремого `GROUP BY`. Найактивніший покупець має 17 замовлень, останнє — 2018-08-20; за Data-quality summary, повторних покупців лише 2 997 із 96 096 (≈3%).

### Data-quality summary

**Business-question:** Які проблеми якості є в typed-таблицях Olist?

In [28]:
%%sql
SELECT 'products without category' AS check_name, COUNT(*) AS n_rows FROM olist_products WHERE product_category_name IS NULL
UNION ALL SELECT 'orders without items', COUNT(*) FROM olist_orders AS o
    WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id)
UNION ALL SELECT 'orders without review', COUNT(*) FROM olist_orders AS o
    WHERE NOT EXISTS (SELECT 1 FROM olist_order_reviews AS rv WHERE rv.order_id = o.order_id)
UNION ALL SELECT 'orders with 2+ reviews', COUNT(*) FROM (
    SELECT order_id FROM olist_order_reviews GROUP BY order_id HAVING COUNT(*) > 1) AS t
UNION ALL SELECT 'duplicated review_id rows', COUNT(*) - COUNT(DISTINCT review_id) FROM olist_order_reviews
UNION ALL SELECT 'reviews without message', COUNT(*) FROM olist_order_reviews WHERE review_comment_message IS NULL
UNION ALL SELECT 'delivered without delivery date', COUNT(*) FROM olist_orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
UNION ALL SELECT 'orders with 2+ sellers', COUNT(*) FROM (
    SELECT order_id FROM olist_order_items GROUP BY order_id HAVING COUNT(DISTINCT seller_id) > 1) AS t
UNION ALL SELECT 'customers with 2+ orders', COUNT(*) FROM (
    SELECT c.customer_unique_id FROM olist_customers AS c
    JOIN olist_orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id HAVING COUNT(*) > 1) AS t;

check_name,n_rows
products without category,610
reviews without message,58247
orders with 2+ reviews,547
delivered without delivery date,8
duplicated review_id rows,814
customers with 2+ orders,2997
orders with 2+ sellers,1278
orders without items,775
orders without review,768


**Interpretation:** 610 товарів без категорії, 775 замовлень без позицій і 768 без відгуку. 547 замовлень мають кілька відгуків, а 814 рядків повторюють уже наявний `review_id`, тому він не може бути PK. 58 247 відгуків (≈59%) без тексту, 8 `delivered`-замовлень без дати доставки. 1 278 замовлень містять товари кількох продавців, а повторних покупців лише 2 997.

## Reflection

Найскладнішим був не конкретний тип `JOIN`, а контроль грануляності: будь-який join `orders` з `order_items` і `reviews` множить рядки. Тому `seller_score` рахується лише після зведення до `seller_id × order_id` і окремої агрегації відгуків до `order_id`, інакше замовлення з кількома позиціями отримали б більшу вагу. З типів найбільше уваги потребував `FULL OUTER JOIN`: потрібно правильно поєднати ключі через `COALESCE` і пояснити `NULL` з кожного боку.

Найважче було сформулювати business-question для `SELF JOIN`: пари продавців корисні для регіонального порівняння, але без додаткових метрик це радше технічна перевірка.

Для production-ETL найкорисніший `ON CONFLICT`: він робить перерахунок ідемпотентним. `RETURNING` поруч із ним дає audit-log без зайвого запиту. Identity-ключі та constraints теж важливі, але вони захищають структуру даних, а не повторюваність процесу.

Window functions я б використав для останнього замовлення покупця, номера покупки, інтервалу між покупками через `LAG()`, ковзного рейтингу продавця і рангу товарів у категорії.

Проблеми якості: 610 товарів без категорії, 768 замовлень без відгуку, 547 замовлень із кількома відгуками, неунікальний `review_id`, 59% відгуків без тексту, 775 замовлень без позицій, 8 `delivered` без дати доставки, ненормалізовані назви міст.

`customer_unique_id` використано, бо `customer_id` створюється для кожного замовлення, і без нього повторні покупки не видно.

Грануляність `seller_id × order_id` потрібна, щоб один відгук не рахувався кілька разів через кілька позицій одного продавця в замовленні.